# Bitcoin Long Accumulation Strategy: Composite On-Chain Signal Index

Below is an implementation of **composite on-chain signal index** to build a long-only, budget-constrained Bitcoin accumulation strategy. The goal is to check if this strategy outperforms uniform Dollar-Cost Averaging (DCA).

### Signal Dimensions

| Signal | Dimension | When to buy |
|---|---|---|
| `mvrv` | Market/realised-value ratio - valuation | Low mvrv = market undervaluation |
| `sopr_7d_ema`| Realised profit/loss of spending | sopr < 1 = investors selling at loss |
| `reserve_risk` | Holder conviction vs. price | Low reserve risk = high confidence, good buy zone |
| `greed_index` | Crowd sentiment | 0 = extreme fear, potential bottoms |
| `puell_multiple` | Miner economics | Low puell multiple = miner stress/capitulation |
| `lth_nupl` | Long-term holder unrealised P&L | Low lth nupl = capitualation/undervaluation |
| `sell_side_risk_ratio_7d_ema` | Short-term selling pressure | Low = market participants not eager to sell |
| `net_unrealized_pnl_rel_to_market_cap` | Aggregate unrealised P&L (NUPL) | Low = investors under water |

### Strategy Logic

Each of **8 on-chain signals** is rolling-z-scored (365-day window, fully causal), inverted into a *cheapness score* (the strategy wants a high score on cheap days. But cheap days produce negative z-scores. So simply negating flips the relationship), and blended via a **softmax-weighted sum**:

$$\text{cheapness}_t = \sum_{i=1}^{8} w_i \cdot \left(-z_t^{(i)}\right), \qquad w_i = \frac{e^{\ell_i}}{\sum_j e^{\ell_j}}$$

$$\text{buy\_score}_t = \max(0,\; \text{cheapness}_t)^{\gamma}$$

The logit vector $\boldsymbol{\ell} \in \mathbb{R}^{8}$ and exponent $\gamma > 0$ are jointly optimised by **Nelder-Mead** on the training period to maximise sats-per-dollar (SPD) relative to uniform DCA.

### Core Constraints
- **365-day rolling window** = budget resets to 1.0 every 365 days (non-overlapping).
- **Sum of weights = 1** per window (fixed total budget).
- **Causal execution** = weight at day $t$ uses only data up to day $t$.
- **Per-day bounds** = $w_t \in [10^{-5},\; 0.1]$.
- **Benchmark** = Uniform DCA: $w_t = 1/365$ for all $t$.

In [1]:
import sys
import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import softmax as scipy_softmax
from scipy.optimize import minimize
from pathlib import Path
from matplotlib import pyplot as plt

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config, data_utils, plots, strategy_utils, analysis

TRAIN_PATH = config.TRAIN_PATH
#STACKSATS_DATA_PATH = config.STACKSATS_DATA_PATH
RAW_PATH = config.RAW_PATH
total_budget_usd = config.TOTAL_BUDGET_USD

#data_utils.check_stacksats_data(STACKSATS_DATA_PATH, RAW_PATH)

# --- Configuration & Time Boundaries ---
TRAIN_START = "2018-01-01"
TRAIN_END   = "2023-12-31"
TEST_START  = "2024-01-01"
TEST_END    = "2025-12-31"
WINDOW_SIZE = 365
MIN_WEIGHT  = 1e-5
MAX_WEIGHT  = 0.1

# --- Signal columns (8 on-chain metrics to weight) ---
SIGNAL_COLS = [
    "mvrv",
    # "adjusted_sopr", 
    # "adjusted_sopr_7d_ema",
    # "realized_cap_growth_rate",
    # "market_cap_growth_rate",
    "sopr_7d_ema",
    "reserve_risk",
    "greed_index",
    "puell_multiple",
    "lth_nupl",
    "sell_side_risk_ratio_7d_ema",
    "net_unrealized_pnl_rel_to_market_cap",
]
N_SIGNALS = len(SIGNAL_COLS)
Z_COLS    = [f"z_{c}" for c in SIGNAL_COLS]

# Metrics needed from parquet (include market_cap + supply_btc to derive price)
LOAD_METRICS = (
    "market_cap",
    "supply_btc",
    *SIGNAL_COLS,
)

# --- Load training data ---
schema  = pl.read_parquet_schema(TRAIN_PATH)
columns = set(schema)

df = (
    pl.scan_parquet(TRAIN_PATH)
    .filter(pl.col("metric").is_in(LOAD_METRICS))
    .select("day_utc", "metric", "value")
    .collect()
    .pivot(values="value", index="day_utc", on="metric")
    .with_columns((pl.col("market_cap") / pl.col("supply_btc")).alias("price_usd"))
    .rename({"day_utc": "date"})
    .select(["date", "price_usd"] + SIGNAL_COLS)
    .filter(pl.col("price_usd").is_finite() & (pl.col("price_usd") > 0))
    .sort("date")
)

print(f"Train rows loaded: {len(df)}")
df.head()

Train rows loaded: 4886


date,price_usd,mvrv,sopr_7d_ema,reserve_risk,greed_index,puell_multiple,lth_nupl,sell_side_risk_ratio_7d_ema,net_unrealized_pnl_rel_to_market_cap
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2010-08-16,0.06,75.25938,0.644378,0.029851,0.0,364.99997,55.39583,21.301033,98.671234
2010-08-17,0.07,56.812218,0.860891,0.033654,0.01,184.63878,55.45579,22.381466,98.23979
2010-08-18,0.07,37.07454,1.048291,0.028037,0.01,125.12996,47.554413,20.86954,83.40232
2010-08-19,0.07,32.22305,1.152911,0.027273,0.01,94.20886,47.629772,17.275007,83.05423
2010-08-20,0.07,28.2621,1.161407,0.026549,0.01,98.38138,47.659138,13.974726,82.681435


## Step 1 - Data Preprocessing: Causal Rolling Z-Scores

Each on-chain signal is standardised using a **causal rolling 365-day z-score**:

$$z_t^{(i)} = \frac{x_t^{(i)} - \mu_{t,365}^{(i)}}{\sigma_{t,365}^{(i)} + \varepsilon}$$

where $\mu_{t,365}$ and $\sigma_{t,365}$ are the mean and standard deviation computed over the trailing 365-day window ending at day $t$ (`min_periods=2` to handle the warm-up period). A small constant $\varepsilon = 10^{-8}$ prevents division by zero.

- **Train period**: 2018-01-01 -> 2023-12-31  
- **Test period**: 2024-01-01 -> 2025-12-31  
- Z-scores are computed on the **combined** dataset so that the rolling window is continuous across the train/test boundary (no data leakage - each day's z-score only uses its own trailing window).

In [2]:
# Load, pivot, and derive price from long-format parquet
def load_and_pivot(path: str) -> pd.DataFrame:
    """Load a long-format parquet, pivot to wide, derive price_usd."""
    return (
        pl.scan_parquet(path)
        .filter(pl.col("metric").is_in(LOAD_METRICS))
        .select("day_utc", "metric", "value")
        .collect()
        .pivot(values="value", index="day_utc", on="metric")
        .with_columns((pl.col("market_cap") / pl.col("supply_btc")).alias("price_usd"))
        .rename({"day_utc": "date"})
        .select(["date", "price_usd"] + SIGNAL_COLS)
        .filter(pl.col("price_usd").is_finite() & (pl.col("price_usd") > 0))
        .sort("date")
        .to_pandas()
    )

# Causal rolling 365-day z-score 
def rolling_zscore(series: pd.Series, window: int = 365) -> pd.Series:
    """Compute a causal rolling z-score using a trailing window of 'window' days. 
    A small epsilon (1e-8) prevents division by zero"""
    roll = series.rolling(window=window, min_periods=2)
    return (series - roll.mean()) / (roll.std() + 1e-8)

# def trim_full_windows(df: pd.DataFrame, window_size: int = WINDOW_SIZE):
#     n_full = len(df) // window_size
#     n_eval = n_full * window_size
#     remainder = len(df) - n_eval
#     return df.iloc[:n_eval].copy(), n_full, remainder

combined_raw = load_and_pivot(RAW_PATH)
combined_raw["date"] = pd.to_datetime(combined_raw["date"])

print(f"Combined dataset: {len(combined_raw)} rows  "
      f"({combined_raw['date'].min().date()} → {combined_raw['date'].max().date()})")
print(f"Missing values per signal:\n{combined_raw[SIGNAL_COLS].isna().sum().to_string()}")

combined = combined_raw.copy()
for col in SIGNAL_COLS:
    combined[f"z_{col}"] = rolling_zscore(combined[col], window=WINDOW_SIZE)

# Forward-fill to handle sparse signals (e.g. reserve_risk gaps), Strict causality: no backward fill
combined[Z_COLS] = combined[Z_COLS].ffill()

# Train / Test splits
train_df = combined[
    (combined["date"] >= TRAIN_START) & (combined["date"] <= TRAIN_END)
].reset_index(drop=True)

test_df = combined[
    (combined["date"] >= TEST_START) & (combined["date"] <= TEST_END)
].reset_index(drop=True)

# Keep rows that are fully usable for scoring
train_df = train_df.dropna(subset=["price_usd"] + Z_COLS).sort_values("date").reset_index(drop=True)
test_df  = test_df.dropna(subset=["price_usd"] + Z_COLS).sort_values("date").reset_index(drop=True)

print(f"NaN in z-scored train signals: {train_df[Z_COLS].isna().sum().sum()}")
train_df[["date", "price_usd"] + Z_COLS].head(10)

Combined dataset: 5689 rows  (2010-08-16 → 2026-03-13)
Missing values per signal:
mvrv                                    0
sopr_7d_ema                             0
reserve_risk                            0
greed_index                             0
puell_multiple                          0
lth_nupl                                0
sell_side_risk_ratio_7d_ema             0
net_unrealized_pnl_rel_to_market_cap    0
NaN in z-scored train signals: 0


,date,price_usd,z_mvrv,z_sopr_7d_ema,z_reserve_risk,z_greed_index,z_puell_multiple,z_lth_nupl,z_sell_side_risk_ratio_7d_ema,z_net_unrealized_pnl_rel_to_market_cap
0,2018-01-01,13466.31,-0.018563,-0.453264,1.519986,2.409743,0.874200,-0.240062,0.642310,0.183284
1,2018-01-02,14888.11,0.445743,-0.300535,1.808096,2.157409,1.402588,-0.217997,0.567724,0.559715
2,2018-01-03,15098.14,0.550742,-0.303331,1.857087,2.114028,1.518643,-0.015153,0.403214,0.750146
3,2018-01-04,15144.99,0.519210,-0.182288,1.825269,1.974431,1.649768,0.008468,0.414528,0.720643
4,2018-01-05,16960.01,1.081544,0.180665,2.193030,2.152453,1.304836,0.091369,0.603476,1.114781
5,2018-01-06,17115.00,1.070447,0.247602,2.178842,2.069565,1.868143,0.167870,0.586168,1.115438
6,2018-01-07,16174.22,0.623257,0.529704,1.904783,2.174020,1.665734,0.010288,0.772679,0.757300
7,2018-01-08,15000.00,0.203236,0.067870,1.605367,2.030056,1.415335,-0.011109,0.593532,0.411809
8,2018-01-09,14427.01,-0.013129,-0.044856,1.452929,2.231179,1.313353,0.019846,0.487306,0.229108
9,2018-01-10,14789.07,0.129649,-0.267004,1.518681,1.899709,1.105382,0.128199,0.384306,0.415337


## Step 2 - Composite On-Chain Signal Index

### Formula

**Cheapness score** (high = market is cheap / under-valued):

$$\text{cheapness}_t = \sum_{i=1}^{8} w_i \cdot \left(-z_t^{(i)}\right)$$

Signals that measure *market heat* (e.g. MVRV, SOPR, greed index) have **positive** z-scores when the market is expensive, so their negation flips them into cheapness contributions.

**Buy score** (non-negative, amplified by exponent):

$$\text{buy\_score}_t = \max\!\left(0,\; \text{cheapness}_t\right)^{\gamma}$$

- When $\text{cheapness}_t \leq 0$ (expensive market) the score is zero - no amplification above baseline.  
- $\gamma > 1$ concentrates capital on the *deepest* cheap periods; $\gamma < 1$ smooths contributions.



In [3]:
def compute_scores(df: pd.DataFrame, logits: np.ndarray, exponent: float) -> np.ndarray:
    """
    Compute the composite buy_score for each row in df.

    Parameters
    ----------
    df       : DataFrame with columns z_{signal} for every signal in SIGNAL_COLS.
    logits   : shape (N_SIGNALS,) - raw logit values; softmax gives the signal weights.
    exponent : float > 0 - sharpness of the buy score.

    Returns
    -------
    buy_score : np.ndarray shape (len(df),), values >= 0.
    """
    weights   = scipy_softmax(logits)                     # (8,) - sums to 1
    z_matrix  = df[Z_COLS].fillna(0.0).to_numpy()         # (N, 8)
    cheapness = z_matrix @ (-weights)                     # (N,) - high = cheap
    buy_score = np.maximum(0.0, cheapness) ** exponent
    return buy_score


# Sanity check with uniform logits (equal weights) and exponent=1 
_logits0   = np.zeros(N_SIGNALS)
_exponent0 = 1.0
_scores0   = compute_scores(train_df, _logits0, _exponent0)

print(f"Uniform-weight buy scores on train set:")
print(f"  Non-zero fraction : {(_scores0 > 0).mean():.2%}")
print(f"  Mean (non-zero)   : {_scores0[_scores0 > 0].mean():.4f}")
print(f"  Max               : {_scores0.max():.4f}")
print(f"  Min (non-zero)    : {_scores0[_scores0 > 0].min():.6f}")


Uniform-weight buy scores on train set:
  Non-zero fraction : 57.33%
  Mean (non-zero)   : 0.9506
  Max               : 2.2538
  Min (non-zero)    : 0.001674


## Step 3 - Online Causal Budget Allocation

### Constraints
| Constraint | Value |
|---|---|
| Sum of weights per window | $= 1$ |
| Min daily weight | $\geq 10^{-5}$ |
| Max daily weight | $\leq 0.1$ |
| Past weights | Immutable once set |
| Future information | Never used |

### Algorithm (per 365-day window)

```
remaining_budget ← 1.0
running_scores   ← []

for t = 0 … 364:
    score_t  ← buy_score[t]          # causal: only uses data ≤ t
    running_scores.append(score_t)
    remaining_days_after ← 364 − t

    if t == 364:                      # last day - consume all remaining budget
        w_t ← remaining_budget

    else:
        running_mean ← mean(running_scores)
        estimated_remaining_sum ← score_t + running_mean × remaining_days_after

        if estimated_remaining_sum > 0:
            w_t_raw ← (score_t / estimated_remaining_sum) × remaining_budget
        else:
            w_t_raw ← remaining_budget / (remaining_days_after + 1)   # uniform fallback

        max_allowed ← min(MAX_WEIGHT, remaining_budget − remaining_days_after × MIN_WEIGHT)
        w_t ← clamp(w_t_raw, MIN_WEIGHT, max(max_allowed, MIN_WEIGHT))

    remaining_budget −= w_t

# Post-process: iterative capping projection to enforce [MIN_WEIGHT, MAX_WEIGHT]
# and renormalise to sum = 1 exactly.
```

**Key insight** - the denominator `score_t + running_mean × remaining_days_after` estimates the expected total score of the remaining window using only past observations (running mean). This is the simplest unbiased estimator available at day $t$ without look-ahead.

The `max_allowed` guard ensures that after taking $w_t$, sufficient budget remains to give every future day at least `MIN_WEIGHT`.

In [4]:
def _project_weights(w: np.ndarray, min_w: float, max_w: float, eps: float = 1e-12) -> np.ndarray:
    """
    Project onto bounded simplex:
      sum(w)=1, min_w <= w_i <= max_w
    with early infeasibility checks and no final renormalization.
    """
    x = np.asarray(w, dtype=float).copy()
    n = x.size

    if n == 0:
        raise ValueError("Empty window.")
    if n * max_w < 1 - eps: 
        raise ValueError(f"Infeasible bounds: n*max_w={n*max_w:.6f} < 1.")
    if n * min_w > 1 + eps:
        raise ValueError(f"Infeasible bounds: n*min_w={n*min_w:.6f} > 1.")

    lo = np.full(n, min_w, dtype=float)
    hi = np.full(n, max_w, dtype=float)
    x = np.clip(x, lo, hi)

    fixed = np.zeros(n, dtype=bool)

    for _ in range(n + 10):
        free = ~fixed # bitwise NOT to get free indices        
        target = 1.0 - x[fixed].sum()

        if target < -eps:
            raise ValueError("Infeasible during projection: fixed weights exceed budget.")
        if free.sum() == 0:
            break

        base = x[free]
        base_sum = base.sum()
        if base_sum <= eps:
            base = np.full(free.sum(), 1.0 / free.sum())
        else:
            base = base / base_sum

        x[free] = base * target

        low = x < lo - eps
        high = x > hi + eps
        violated = low | high
        if not violated.any():
            break

        x[low] = lo[low]
        x[high] = hi[high]
        fixed[low | high] = True

    # Final boundedness and exact-sum checks (no renormalization)
    if (x < lo - 1e-9).any() or (x > hi + 1e-9).any():
        raise ValueError("Projection failed bounds check.")
    if abs(x.sum() - 1.0) > 1e-8:
        # small correction through free-slack only
        resid = 1.0 - x.sum()
        slack = (x > lo + 1e-9) & (x < hi - 1e-9)
        if slack.any():
            x[slack] += resid / slack.sum()
        if (x < lo - 1e-9).any() or (x > hi + 1e-9).any() or abs(x.sum() - 1.0) > 1e-8:
            raise ValueError(f"Projection failed exact-sum check: sum={x.sum():.12f}")

    return x

def allocate_weights(
    scores: np.ndarray,
    window_size: int = WINDOW_SIZE,
    min_w: float = MIN_WEIGHT,
    max_w: float = MAX_WEIGHT,
) -> np.ndarray:
    """
    Ignore remainder everywhere: input length must be exactly multiple of window_size.
    """
    scores = np.asarray(scores, dtype=float)
    if (scores < 0).any():
        raise ValueError("Scores must be non-negative.")

    N = len(scores)
    if N % window_size != 0:
        raise ValueError(
            f"Scores length {N} not divisible by window_size={window_size}. "
            f"Trim to full windows before calling."
        )

    n_windows = N // window_size
    weights = np.zeros(N, dtype=float)

    for win_idx in range(n_windows):
        start = win_idx * window_size
        end = start + window_size
        win_scores = scores[start:end]

        remaining_budget = 1.0
        win_weights = np.zeros(window_size, dtype=float)
        running_sum = 0.0

        for i, score_t in enumerate(win_scores):
            remaining_days_after = window_size - 1 - i
            running_sum += score_t
            running_mean = running_sum / (i + 1)

            if i == window_size - 1:
                w_t = remaining_budget
            else:
                est_remaining = score_t + running_mean * remaining_days_after
                if est_remaining > 1e-12:
                    w_t_raw = (score_t / est_remaining) * remaining_budget
                else:
                    w_t_raw = remaining_budget / (remaining_days_after + 1)

                max_allowed = min(max_w, remaining_budget - remaining_days_after * min_w)
                max_allowed = max(max_allowed, min_w)
                w_t = float(np.clip(w_t_raw, min_w, max_allowed))

            win_weights[i] = w_t
            remaining_budget = max(remaining_budget - w_t, 0.0)

        weights[start:end] = _project_weights(win_weights, min_w, max_w)
        
    return weights

def verify_constraints_exact_eval(
    weights: np.ndarray,
    n_windows: int,
    window_size: int = WINDOW_SIZE,
    min_w: float = MIN_WEIGHT,
    max_w: float = MAX_WEIGHT,
    label: str = "",
    tol: float = 1e-8,
) -> None:
    """
    Verify exactly and only the evaluated full-window set.
    """
    expected_len = n_windows * window_size
    if len(weights) != expected_len:
        raise ValueError(f"{label} expected len={expected_len}, got {len(weights)}")

    errors = []
    for w in range(n_windows):
        s = weights[w * window_size:(w + 1) * window_size]
        if abs(s.sum() - 1.0) > tol:
            errors.append(f"Window {w+1}: sum={s.sum():.12f}")
        if s.min() < min_w - tol:
            errors.append(f"Window {w+1}: min={s.min():.12f} < {min_w}")
        if s.max() > max_w + tol:
            errors.append(f"Window {w+1}: max={s.max():.12f} > {max_w}")

    tag = f"[{label}] " if label else ""
    if errors:
        raise AssertionError(tag + "Constraint violations:\n" + "\n".join(errors))
    print(f"{tag}All evaluated windows pass constraints.")


## Step 4 - Nelder-Mead Optimization

### Parameter Space

| Parameter | Representation | Count |
|---|---|---|
| Signal logits $\ell_1 \dots \ell_{12}$ | Unconstrained reals; signal weights $= \text{softmax}(\boldsymbol{\ell})$ | 8 |
| Log-exponent $\log \gamma$ | Unconstrained real; $\gamma = e^{\log\gamma}$ clipped to $[e^{-3}, e^{3}]$ | 1 |
| **Total** |- | **9** |

Starting point $\boldsymbol{\theta}_0 = \mathbf{0}$ corresponds to **equal signal weights** and **$\gamma = 1$**.

### Objective

$$\max_{\boldsymbol{\theta}} \;\frac{1}{K}\sum_{k=1}^{K} \frac{\text{SPD}_{\text{strategy}}^{(k)}}{\text{SPD}_{\text{DCA}}^{(k)}}$$

where $K$ is the number of complete 365-day windows in the training period, and

$$\text{SPD}^{(k)} = \sum_{t \in \text{window } k} \frac{w_t}{P_t}$$

is the **sats-per-dollar** accumulated in window $k$ (higher = better).
Implemented as `minimize(-objective, θ, method="Nelder-Mead")`.

In [5]:
# Build train_eval_df year-by-year, matching export_one_year's leap-year logic:
# Leap years (366 rows) → use Jan 2 – Dec 31 (drop Jan 1 = first row) → exactly 365 rows
_year_slices = []
for year, grp in train_df.groupby(train_df["date"].dt.year):
    grp = grp.sort_values("date").reset_index(drop=True)
    if len(grp) >= WINDOW_SIZE:
        _year_slices.append(grp.iloc[len(grp) - WINDOW_SIZE:])  # last 365 rows

train_eval_df    = pd.concat(_year_slices, ignore_index=True)
_n_windows_train = len(train_eval_df) // WINDOW_SIZE   # == number of complete calendar years
_train_prices    = train_eval_df["price_usd"].to_numpy()
_dca_w           = np.full(WINDOW_SIZE, 1.0 / WINDOW_SIZE)

print(f"window size: {WINDOW_SIZE}, length of train_df: {len(train_df)}, "
      f"train_eval rows: {len(train_eval_df)}, number of windows: {_n_windows_train}")

def _spd_ratio(weights: np.ndarray, prices: np.ndarray, n_windows: int) -> float:
    """Mean per-window SPD ratio: strategy / uniform-DCA."""
    ratios = np.empty(n_windows, dtype=float)
    for k in range(n_windows):
        sl = slice(k * WINDOW_SIZE, (k + 1) * WINDOW_SIZE)
        p  = prices[sl]
        inv_p = 1.0 / p
        spd_strat = np.dot(weights[sl], inv_p)
        spd_dca   = np.dot(_dca_w,      inv_p)
        ratios[k] = spd_strat / (spd_dca + 1e-15)
    return float(ratios.mean())


_eval_counter = [0]


def objective(theta: np.ndarray) -> float:
    """Nelder-Mead objective - returns negative mean SPD ratio (to minimise)."""
    logits   = theta[:N_SIGNALS]
    exponent = float(np.exp(np.clip(theta[N_SIGNALS], -3.0, 3.0)))

    scores  = compute_scores(train_eval_df, logits, exponent)   # <-- trimmed df
    weights = allocate_weights(scores, window_size=WINDOW_SIZE, min_w=MIN_WEIGHT, max_w=MAX_WEIGHT)

    ratio = _spd_ratio(weights, _train_prices, _n_windows_train)

    _eval_counter[0] += 1
    if _eval_counter[0] % 500 == 0:
        print(f"  eval {_eval_counter[0]:>5d} | SPD ratio {ratio:.6f} | γ={exponent:.4f}")

    return -ratio


# Run Nelder-Mead
print("Starting Nelder-Mead optimisation (up to 5 000 function evaluations)…\n")
theta0 = np.zeros(N_SIGNALS + 1)

result = minimize(
    objective,
    theta0,
    method="Nelder-Mead",
    options={
        "maxfev"  : 5000,
        "xatol"   : 1e-4,
        "fatol"   : 1e-4,
        "adaptive": True,
    },
)

opt_logits   = result.x[:N_SIGNALS]
opt_exponent = float(np.exp(np.clip(result.x[N_SIGNALS], -3.0, 3.0)))
opt_sig_w    = scipy_softmax(opt_logits)

print(f"\nOptimisation complete")
print(f"  Status              : {result.message}")
print(f"  Function evals      : {result.nfev}")
print(f"  Converged           : {result.success}")
print(f"  Best SPD ratio      : {-result.fun:.6f}  (+{(-result.fun - 1)*100:.2f}% vs DCA)")
print(f"  Optimal logits      : {opt_logits}")
print(f"  Optimal exponent γ  : {opt_exponent:.4f}")
print(f"\nLearned signal weights (softmax):")
for col, w in zip(SIGNAL_COLS, opt_sig_w):
    print(f"  {col:<45s} {w:.4f}")

window size: 365, length of train_df: 2191, train_eval rows: 2190, number of windows: 6
Starting Nelder-Mead optimisation (up to 5 000 function evaluations)…

  eval   500 | SPD ratio 1.282699 | γ=20.0855
  eval  1000 | SPD ratio 1.287466 | γ=20.0855

Optimisation complete
  Status              : Optimization terminated successfully.
  Function evals      : 1494
  Converged           : True
  Best SPD ratio      : 1.290656  (+29.07% vs DCA)
  Optimal logits      : [ 0.6879027  -0.15186923 -1.63686429 -0.4787459  -0.7271734   0.44625924
 -2.33481893  1.40813299]
  Optimal exponent γ  : 20.0855

Learned signal weights (softmax):
  mvrv                                          0.2011
  sopr_7d_ema                                   0.0868
  reserve_risk                                  0.0197
  greed_index                                   0.0626
  puell_multiple                                0.0488
  lth_nupl                                      0.1579
  sell_side_risk_ratio_7d_ema      

In [6]:
# Learned signal weights bar chart
baseline_w = 1.0 / N_SIGNALS   # equal-weight reference

fig_w = go.Figure()
fig_w.add_trace(go.Bar(
    y=SIGNAL_COLS,
    x=opt_sig_w,
    orientation="h",
    text=[f"{w:.3f}" for w in opt_sig_w],
    textposition="outside",
    marker_color=[
        "steelblue" if w >= baseline_w else "lightcoral"
        for w in opt_sig_w
    ],
    name="Optimised weight",
))
fig_w.add_vline(
    x=baseline_w,
    line_dash="dash",
    line_color="black",
)
fig_w.add_annotation(
    x=baseline_w,
    y=1,              # > 1 pushes above plot area ("outside")
    xref="x",
    yref="paper",
    text=f"<b>Equal weight ({baseline_w:.3f})</b>",
    showarrow=False,     
    xanchor="left",
    yanchor="bottom",
)
fig_w.update_xaxes(
    tickfont=dict(size=16, color="black")
)
fig_w.update_yaxes(
    tickfont=dict(family="Arial Black", size=16, color="black")
)
fig_w.update_layout(
    title=f"Optimised Signal Weights  (γ = {opt_exponent:.3f})",
    yaxis_title="On-chain signal",
    xaxis_title="Weight (softmax)",
    margin=dict(l=220, r=40, t=70, b=40),
    showlegend=False,
)
fig_w.update_yaxes(autorange="reversed")
fig_w.show()

## Step 5 - Backtest

We evaluate the strategy on the training set using the optimised parameters.  
**SPD improvement** per window = `(strategy_SPD / DCA_SPD) − 1`.  
A positive value means the strategy accumulated **more BTC per dollar spent** than uniform DCA.

In [7]:
# Build test_eval_df year-by-year, matching export_one_year's leap-year logic:
# Leap years (366 rows) → use Jan 2 – Dec 31 (drop Jan 1 = first row) → exactly 365 rows
_test_year_slices = []
for year, grp in test_df.groupby(test_df["date"].dt.year):
    grp = grp.sort_values("date").reset_index(drop=True)
    if len(grp) >= WINDOW_SIZE:
        _test_year_slices.append(grp.iloc[len(grp) - WINDOW_SIZE:])  # last 365 rows

test_eval_df = pd.concat(_test_year_slices, ignore_index=True)

print(f"test_df rows: {len(test_df)}, test_eval rows: {len(test_eval_df)}, "
      f"windows: {len(test_eval_df) // WINDOW_SIZE}")

test_df rows: 731, test_eval rows: 730, windows: 2


In [8]:
# use stacksats backtester for final evaluation on train + test sets
from stacksats import StrategyRunner, BacktestConfig
from stacksats.strategy_types import BaseStrategy, DayState

class _CSIStrategy(BaseStrategy):
    """Thin wrapper to expose opt_logits/opt_exponent via propose_weight."""
    strategy_id = "composite-signal-index"
    version     = "1.0.0"

    def __init__(self, logits, exponent, zscore_lookup, min_w, max_w, window_size):
        self.opt_logits   = logits
        self.opt_exponent = exponent
        self._lookup      = zscore_lookup
        self.min_weight   = min_w
        self.max_weight   = max_w
        self.window_size  = window_size
        self._running_sum = 0.0

    def params(self):
        return {"opt_logits": self.opt_logits.tolist(), "opt_exponent": self.opt_exponent}

    def propose_weight(self, state: DayState) -> float:
        day_idx  = state.day_index
        total    = state.total_days
        remaining = float(state.remaining_budget)

        if day_idx == 0:
            self._running_sum = 0.0

        current_date = pd.Timestamp(state.features.select("date").tail(1).item())
        z_vec = self._lookup.get(current_date)

        if z_vec is None:
            remaining_days = total - day_idx
            return float(np.clip(remaining / remaining_days, self.min_weight, self.max_weight))

        sig_w     = scipy_softmax(self.opt_logits)
        cheapness = float(np.dot(-z_vec, sig_w))
        score     = float(max(0.0, cheapness) ** self.opt_exponent)

        self._running_sum += score
        running_mean         = self._running_sum / (day_idx + 1)
        remaining_after      = total - 1 - day_idx

        if day_idx == total - 1:
            return float(remaining)

        est = score + running_mean * remaining_after
        w_raw = (score / est) * remaining if est > 1e-12 else remaining / (remaining_after + 1)
        max_allowed = max(min(self.max_weight, remaining - remaining_after * self.min_weight), self.min_weight)
        return float(np.clip(w_raw, self.min_weight, max_allowed))

# Build date → z-vector lookup from combined df
zscore_lookup = {
    pd.Timestamp(row["date"]): row[Z_COLS].to_numpy(dtype=float)
    for _, row in combined.iterrows()
}

csi_strategy = _CSIStrategy(
    logits       = opt_logits,
    exponent     = opt_exponent,
    zscore_lookup= zscore_lookup,
    min_w        = MIN_WEIGHT,
    max_w        = MAX_WEIGHT,
    window_size  = WINDOW_SIZE,
)

# Build btc_df for runner (train + test dates)
full_eval_df = pd.concat([train_eval_df, test_eval_df], ignore_index=True)
btc_df_pl = (
    pl.from_pandas(full_eval_df[["date", "price_usd"]].assign(
        date=lambda d: pd.to_datetime(d["date"]).dt.date
    ))
    .with_columns(pl.col("date").cast(pl.Date))
)

runner     = StrategyRunner()
start_date = str(full_eval_df["date"].min().date())
end_date   = str(full_eval_df["date"].max().date())

csi_bt = runner.backtest(
    csi_strategy,
    config=BacktestConfig(start_date=start_date, end_date=end_date),
    btc_df=btc_df_pl,
)
print(csi_bt.to_dataframe())

shape: (2_558, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ window     ┆ min_sats_p ┆ max_sats_p ┆ uniform_s ┆ dynamic_s ┆ uniform_p ┆ dynamic_p ┆ excess_pe │
│ ---        ┆ er_dollar  ┆ er_dollar  ┆ ats_per_d ┆ ats_per_d ┆ ercentile ┆ ercentile ┆ rcentile  │
│ str        ┆ ---        ┆ ---        ┆ ollar     ┆ ollar     ┆ ---       ┆ ---       ┆ ---       │
│            ┆ f64        ┆ f64        ┆ ---       ┆ ---       ┆ f64       ┆ f64       ┆ f64       │
│            ┆            ┆            ┆ f64       ┆ f64       ┆           ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2018-01-01 ┆ 5842.82792 ┆ 31422.7268 ┆ 14736.064 ┆ 18730.013 ┆ 34.766505 ┆ 50.380128 ┆ 15.613623 │
│ →          ┆ 9          ┆ 01         ┆ 632       ┆ 691       ┆           ┆           ┆           │
│ 2018-12-31 ┆            ┆            ┆           ┆           ┆         

In [9]:
from stacksats import UniformStrategy
from src import strategy_utils

csi_yearly      = []
uniform_yearly  = []
uniform_strategy = UniformStrategy()

all_years = sorted(full_eval_df["date"].dt.year.unique())
for year in all_years:
    csi_yr = strategy_utils.export_one_year(csi_strategy, btc_df_pl, year, runner)
    if csi_yr is not None:
        csi_yearly.append(csi_yr)
    uni_yr = strategy_utils.export_one_year(uniform_strategy, btc_df_pl, year, runner)
    if uni_yr is not None:
        uniform_yearly.append(uni_yr)

csi_all     = pl.concat(csi_yearly).rename({"weight": "csi_weight"})
uniform_all = pl.concat(uniform_yearly).rename({"weight": "baseline_weight"})

merged_runner = (
    csi_all.select(["date", "price_usd", "csi_weight"])
    .join(uniform_all.select(["date", "baseline_weight"]), on="date", how="inner")
    .sort("date")
    .with_columns(pl.col("date").cast(pl.Date).dt.year().alias("year"))
)

merged_runner = merged_runner.with_columns([
    (pl.col("csi_weight")      * total_budget_usd).alias("csi_usd"),
    (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
]).with_columns([
    (pl.col("csi_usd")      / pl.col("price_usd")).alias("btc_accum_csi"),
    (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
]).with_columns([
    (pl.col("btc_accum_csi")      * 100_000_000).alias("sats_accum_csi"),
    (pl.col("btc_accum_baseline") * 100_000_000).alias("sats_accum_baseline"),
])

# Per-year SPD improvement (Panel 4 equivalent)
spd_by_year = (
    merged_runner
    .group_by("year")
    .agg([
        (pl.col("csi_weight")      / pl.col("price_usd")).sum().alias("spd_csi"),
        (pl.col("baseline_weight") / pl.col("price_usd")).sum().alias("spd_baseline"),
    ])
    .with_columns(((pl.col("spd_csi") / pl.col("spd_baseline") - 1) * 100).alias("improvement_pct"))
    .sort("year")
)

total_dynamic_btc  = merged_runner["btc_accum_csi"].sum()
total_baseline_btc = merged_runner["btc_accum_baseline"].sum()
n_years            = merged_runner["year"].n_unique()
total_spend_usd    = total_budget_usd * n_years

sats_per_dollar_dynamic  = (total_dynamic_btc  / total_spend_usd) * 100_000_000
sats_per_dollar_baseline = (total_baseline_btc / total_spend_usd) * 100_000_000
pct_diff = (total_dynamic_btc - total_baseline_btc) / total_baseline_btc * 100
label    = "better" if pct_diff > 0 else "worse"

print(f"Total BTC, CSI:     {total_dynamic_btc:.6f}  (${total_spend_usd:.0f} deployed)")
print(f"Total BTC, Uniform: {total_baseline_btc:.6f}")
print(f"SPD, CSI:           {sats_per_dollar_dynamic:.2f}")
print(f"SPD, Uniform:       {sats_per_dollar_baseline:.2f}")
print(f"CSI {abs(pct_diff):.2f}% {label} than Uniform")
print(spd_by_year)

# --- Recompute buy scores for Panel 1 (batch, from opt params) ---
runner_scores = compute_scores(full_eval_df, opt_logits, opt_exponent)

plot_pd     = merged_runner.to_pandas()
plot_pd["date"] = pd.to_datetime(plot_pd["date"])
runner_dates    = pd.to_datetime(plot_pd["date"].to_numpy())
runner_weights  = plot_pd["csi_weight"].to_numpy()
runner_prices   = plot_pd["price_usd"].to_numpy()
dca_weight      = 1.0 / WINDOW_SIZE

btc_per_day_strat = runner_weights  * total_budget_usd / runner_prices
btc_per_day_dca   = dca_weight      * total_budget_usd / runner_prices
cum_strat_r = np.cumsum(btc_per_day_strat)
cum_dca_r   = np.cumsum(btc_per_day_dca)

# Bar data from spd_by_year
spd_pd      = spd_by_year.to_pandas().sort_values("year")
bar_labels_r = [str(y) for y in spd_pd["year"]]
bar_colors_r = ["steelblue" if y <= 2023 else "seagreen" for y in spd_pd["year"]]

fig2 = make_subplots(
    rows=4, cols=1,
    subplot_titles=[
        "Composite Buy Score (cheapness index)",
        "Daily Allocation Weight - Strategy vs. Uniform DCA",
        "Cumulative BTC Accumulated: Strategy vs. Uniform DCA",
        "Per-Year SPD Improvement over Uniform DCA (%)",
    ],
    vertical_spacing=0.08,
    row_heights=[0.18, 0.18, 0.36, 0.28],
)

fig2.add_trace(go.Scatter(x=runner_dates, y=runner_scores,
    name="Buy score", line=dict(color="goldenrod", width=1)), row=1, col=1)

fig2.add_trace(go.Scatter(x=runner_dates, y=runner_weights,
    name="Strategy weight", line=dict(color="steelblue", width=0.8),
    fill="tozeroy", fillcolor="rgba(70,130,180,0.15)"), row=2, col=1)
fig2.add_trace(go.Scatter(
    x=[runner_dates.min(), runner_dates.max()], y=[dca_weight, dca_weight],
    name=f"Uniform DCA (1/365)", line=dict(color="red", dash="dot", width=1.5)), row=2, col=1)

fig2.add_trace(go.Scatter(x=runner_dates, y=cum_strat_r,
    name="Strategy", line=dict(color="steelblue", width=2)), row=3, col=1)
fig2.add_trace(go.Scatter(x=runner_dates, y=cum_dca_r,
    name="Uniform DCA", line=dict(color="red", dash="dot", width=1.5)), row=3, col=1)

for row_i in [1, 2, 3]:
    fig2.add_vrect(x0=TEST_START, x1=str(runner_dates.max().date()),
        fillcolor="lightgreen", opacity=0.15, line_width=0,
        annotation_text="Test", annotation_position="top left", row=row_i, col=1)

fig2.add_trace(go.Bar(
    x=bar_labels_r, y=spd_pd["improvement_pct"],
    marker_color=bar_colors_r,
    text=[f"{v:.1f}%" for v in spd_pd["improvement_pct"]],
    textposition="outside", name="SPD improvement %"), row=4, col=1)
fig2.add_hline(y=0, line_color="black", line_width=1, row=4, col=1)

fig2.update_layout(
    title=dict(text=(
        f"CSI Strategy - runner.backtest view<br>"
        f"<sup>γ={opt_exponent:.3f}  |  "
        f"SPD: {sats_per_dollar_dynamic:.1f} vs {sats_per_dollar_baseline:.1f} (uniform)  |  "
        f"{abs(pct_diff):.2f}% {label}</sup>"
    ), x=0.5),
    height=1700,
    legend=dict(orientation="h", y=-0.02),
    margin=dict(l=60, r=30, t=100, b=60),
)
fig2.update_yaxes(title_text="Buy score",        row=1, col=1)
fig2.update_yaxes(title_text="Weight",           row=2, col=1)
fig2.update_yaxes(title_text="BTC accumulated",  row=3, col=1)
fig2.update_yaxes(title_text="Improvement %",    row=4, col=1)
fig2.update_xaxes(title_text="Date",             row=4, col=1)
fig2.show()

2018: exported 365 rows
2018: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows
2020: exported 365 rows
2020: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2022: exported 365 rows
2022: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
2024: exported 365 rows
2024: exported 365 rows
2025: exported 365 rows
2025: exported 365 rows
Total BTC, CSI:     0.551705  ($8000 deployed)
Total BTC, Uniform: 0.530970
SPD, CSI:           6896.31
SPD, Uniform:       6637.12
CSI 3.91% better than Uniform
shape: (8, 4)
┌──────┬──────────┬──────────────┬─────────────────┐
│ year ┆ spd_csi  ┆ spd_baseline ┆ improvement_pct │
│ ---  ┆ ---      ┆ ---          ┆ ---             │
│ i32  ┆ f64      ┆ f64          ┆ f64             │
╞══════╪══════════╪══════════════╪═════════════════╡
│ 2018 ┆ 0.000187 ┆ 0.000147     ┆ 27.103227       │
│ 2019 ┆ 0.000151 ┆ 0.000159     ┆ -5.167536       │
│ 2020 ┆ 0.000098 ┆ 0.0001       ┆ -2.563035       │
│ 2021 ┆ 0.00002

|#|_CSIStrategy|CompositeSignalIndex|
|---|---|---|
|Purpose|Thin runner adapter - wraps already-optimized opt_logits/opt_exponent for use with runner.backtest|Full self-contained strategy class - owns data loading, fitting, and inference|
|Fitting|None - takes pre-computed opt_logits and opt_exponent as constructor args|Has fit(train_df) that runs Nelder-Mead internally|
|Z-score lookup|Passed in as a prebuilt dict at construction time|Builds _zscore_lookup internally from full_zscored_df passed to __init__|
|_allocate_weights_batch()|Not present - only propose_weight()|Present - used by fit() for optimization|
|Reusability|Single-use notebook helper - prefixed with _ to signal this|Designed as a reusable, exportable strategy class|
|params()|Returns only logits + exponent|Returns full config including signal_cols, min/max_weight, window_size|

In [10]:
from stacksats.strategy_types import BaseStrategy, DayState

class CompositeSignalIndex(BaseStrategy):
    """
    Composite On-Chain Signal Index - long-only BTC accumulation strategy.

    Each of N on-chain signals is rolling-z-scored (externally, causally),
    blended via softmax-weighted cheapness score, then allocated through a
    causal online budget algorithm within each 365-day window.

    Signal weights (logits) and the sharpness exponent γ are jointly
    optimised by Nelder-Mead on the training period to maximise mean
    sats-per-dollar relative to uniform DCA.

    Parameters
    ----------
    full_zscored_df :
        Combined (train + test) pandas DataFrame with columns
        ``date``, ``price_usd``, and ``z_{signal}`` for every signal.
        Z-scores must already be computed causally (rolling 365-day).
    signal_cols :
        Ordered list of raw signal names (used to derive z-column names).
    min_weight, max_weight :
        Per-day allocation bounds (default 1e-5 / 0.1).
    window_size :
        Budget reset cadence in days (default 365).
    nelder_mead_maxfev :
        Max function evaluations for the optimiser.
    """

    strategy_id  = "composite-signal-index"
    version      = "1.0.0"
    description  = (
        "Softmax-blended on-chain cheapness index with "
        "Nelder-Mead optimised signal weights and sharpness exponent."
    )

    # Construction

    def __init__(
        self,
        full_zscored_df: pd.DataFrame,
        signal_cols: list[str] = SIGNAL_COLS,
        min_weight: float = MIN_WEIGHT,
        max_weight: float = MAX_WEIGHT,
        window_size: int  = WINDOW_SIZE,
        nelder_mead_maxfev: int = 5_000,
    ) -> None:
        self.signal_cols        = list(signal_cols)
        self.z_cols             = [f"z_{c}" for c in self.signal_cols]
        self.n_signals          = len(self.signal_cols)
        self.min_weight         = min_weight
        self.max_weight         = max_weight
        self.window_size        = window_size
        self.nelder_mead_maxfev = nelder_mead_maxfev

        # Store the full z-scored dataset (train + test)
        df = full_zscored_df.copy()
        df["date"] = pd.to_datetime(df["date"])
        self._full_df = df.sort_values("date").reset_index(drop=True)

        # Date → z-vector lookup for propose_weight (O(1) per day)
        self._zscore_lookup: dict[pd.Timestamp, np.ndarray] = {
            row["date"]: row[self.z_cols].to_numpy(dtype=float)
            for _, row in self._full_df.iterrows()
        }

        # Fitted parameters (populated by fit())
        self.opt_logits:   np.ndarray | None = None
        self.opt_exponent: float | None      = None
        self.fit_result_                     = None

        # Running state for propose_weight (reset each window)
        self._running_sum: float = 0.0

    # Fitting

    def fit(self, train_df: pd.DataFrame, verbose: bool = True) -> "CompositeSignalIndex":
        """
        Run Nelder-Mead optimisation on train_df to find the best signal
        logits and exponent γ (maximises mean SPD ratio vs uniform DCA).

        train_df must be a subset of full_zscored_df and include
        ``date``, ``price_usd``, and all ``z_{signal}`` columns.
        Rows are trimmed to an integer multiple of window_size internally.

        Returns self (chainable).
        """
        df = (
            train_df.copy()
            .assign(date=lambda d: pd.to_datetime(d["date"]))
            .dropna(subset=["price_usd"] + self.z_cols)
            .sort_values("date")
            .reset_index(drop=True)
        )

        year_slices = []
        for year, grp in df.groupby(df["date"].dt.year):
            grp = grp.sort_values("date").reset_index(drop=True)
            if len(grp) >= self.window_size:
                year_slices.append(grp.iloc[len(grp) - self.window_size:])
        df = pd.concat(year_slices, ignore_index=True)
        n_full = len(df) // self.window_size

        train_eval   = df.iloc[: n_full * self.window_size].copy()
        train_prices = train_eval["price_usd"].to_numpy()
        dca_w        = np.full(self.window_size, 1.0 / self.window_size)
        _counter     = [0]

        def _objective(theta: np.ndarray) -> float:
            logits   = theta[: self.n_signals]
            exponent = float(np.exp(np.clip(theta[self.n_signals], -3.0, 3.0)))
            scores   = self._compute_scores(train_eval, logits, exponent)
            weights  = self._allocate_weights_batch(scores)

            ratios = np.empty(n_full, dtype=float)
            for k in range(n_full):
                sl      = slice(k * self.window_size, (k + 1) * self.window_size)
                inv_p   = 1.0 / train_prices[sl]
                ratios[k] = np.dot(weights[sl], inv_p) / (np.dot(dca_w, inv_p) + 1e-15)

            _counter[0] += 1
            if verbose and _counter[0] % 500 == 0:
                γ = float(np.exp(np.clip(theta[self.n_signals], -3.0, 3.0)))
                print(f"  eval {_counter[0]:>5d} | SPD ratio {ratios.mean():.6f} | γ={γ:.4f}")

            return -float(ratios.mean())   # minimise → maximise SPD ratio

        if verbose:
            print("Starting Nelder-Mead optimisation …\n")

        result = minimize(
            _objective,
            x0     = np.zeros(self.n_signals + 1),   # uniform logits + γ=1
            method = "Nelder-Mead",
            options = {
                "maxfev"  : self.nelder_mead_maxfev,
                "xatol"   : 1e-4,
                "fatol"   : 1e-4,
                "adaptive": True,
            },
        )

        self.opt_logits   = result.x[: self.n_signals]
        self.opt_exponent = float(np.exp(np.clip(result.x[self.n_signals], -3.0, 3.0)))
        self.fit_result_  = result

        if verbose:
            print(f"\nOptimisation complete")
            print(f"  Status         : {result.message}")
            print(f"  Function evals : {result.nfev}")
            print(f"  Best SPD ratio : {-result.fun:.6f}  (+{(-result.fun - 1)*100:.2f}% vs DCA)")
            print(f"  γ              : {self.opt_exponent:.4f}")
            print(f"\nLearned signal weights (softmax):")
            opt_sig_w = scipy_softmax(self.opt_logits)
            for col, w in zip(self.signal_cols, opt_sig_w):
                print(f"  {col:<45s} {w:.4f}")

        return self

    # BaseStrategy interface

    def params(self) -> dict[str, object]:
        return {
            "signal_cols"       : self.signal_cols,
            "min_weight"        : self.min_weight,
            "max_weight"        : self.max_weight,
            "window_size"       : self.window_size,
            "opt_logits"        : self.opt_logits.tolist() if self.opt_logits is not None else None,
            "opt_exponent"      : self.opt_exponent,
            "nelder_mead_maxfev": self.nelder_mead_maxfev,
        }

    def propose_weight(self, state: DayState) -> float:
        """
        Causal online budget allocation for one day within a 365-day window.

        Called by the stacksats runner day-by-day.  Uses the optimised
        signal logits and exponent to compute today's buy_score, then
        allocates budget proportional to an estimated remaining score sum
        (running mean of observed scores - no look-ahead).

        Raises RuntimeError if fit() has not been called.
        """
        if self.opt_logits is None or self.opt_exponent is None:
            raise RuntimeError("Call fit(train_df) before running the strategy.")

        day_idx    = state.day_index    # 0-based within the current window
        total_days = state.total_days   # == window_size (365)
        remaining  = float(state.remaining_budget)

        # Reset running state at the start of each new window
        if day_idx == 0:
            self._running_sum = 0.0

        # Look up today's z-scores
        current_date = pd.Timestamp(
            state.features.select("date").tail(1).item()
        )
        z_vec = self._zscore_lookup.get(current_date)
        if z_vec is None:
            # Date not found - uniform fallback
            remaining_days = total_days - day_idx
            return float(np.clip(
                remaining / remaining_days,
                self.min_weight, self.max_weight,
            ))

        # Compute today's buy score
        sig_weights = scipy_softmax(self.opt_logits)             # (n_signals,)
        cheapness   = float(np.dot(-z_vec, sig_weights))
        buy_score   = float(max(0.0, cheapness) ** self.opt_exponent)

        self._running_sum += buy_score
        running_mean        = self._running_sum / (day_idx + 1)
        remaining_days_after = total_days - 1 - day_idx

        # Causal budget allocation
        if day_idx == total_days - 1:          # last day: spend remainder
            return float(remaining)

        est_remaining = buy_score + running_mean * remaining_days_after
        if est_remaining > 1e-12:
            w_raw = (buy_score / est_remaining) * remaining
        else:
            w_raw = remaining / (remaining_days_after + 1)

        max_allowed = min(
            self.max_weight,
            remaining - remaining_days_after * self.min_weight,
        )
        max_allowed = max(max_allowed, self.min_weight)

        return float(np.clip(w_raw, self.min_weight, max_allowed))
    
    # Private helpers

    def _compute_scores(
        self,
        df: pd.DataFrame,
        logits: np.ndarray,
        exponent: float,
    ) -> np.ndarray:
        """Vectorised buy-score computation (mirrors notebook compute_scores)."""
        weights   = scipy_softmax(logits)
        z_matrix  = df[self.z_cols].fillna(0.0).to_numpy()
        cheapness = z_matrix @ (-weights)
        return np.maximum(0.0, cheapness) ** exponent
    
    def _allocate_weights_batch(self, scores: np.ndarray) -> np.ndarray:
        """
        Batch causal budget allocation across all full windows.
        Mirrors the notebook's allocate_weights() - used during fit() and
        build_plot_df() only (not during live propose_weight inference).
        """
        scores   = np.asarray(scores, dtype=float)
        N        = len(scores)
        n_wins   = N // self.window_size
        out      = np.zeros(N, dtype=float)

        for w in range(n_wins):
            s      = w * self.window_size
            e      = s + self.window_size
            win_sc = scores[s:e]

            remaining_budget = 1.0
            win_w            = np.zeros(self.window_size, dtype=float)
            running_sum      = 0.0

            for i, score_t in enumerate(win_sc):
                remaining_days_after = self.window_size - 1 - i
                running_sum += score_t
                running_mean = running_sum / (i + 1)

                if i == self.window_size - 1:
                    w_t = remaining_budget
                else:
                    est = score_t + running_mean * remaining_days_after
                    w_raw = (
                        (score_t / est) * remaining_budget
                        if est > 1e-12
                        else remaining_budget / (remaining_days_after + 1)
                    )
                    max_allowed = max(
                        min(self.max_weight, remaining_budget - remaining_days_after * self.min_weight),
                        self.min_weight,
                    )
                    w_t = float(np.clip(w_raw, self.min_weight, max_allowed))

                win_w[i] = w_t
                remaining_budget = max(remaining_budget - w_t, 0.0)

            out[s:e] = _project_weights(win_w, self.min_weight, self.max_weight)
            #out[s:e] = win_w

        return out

In [11]:
from src.plots import StrategyColumns, plot_strategy_full_period, plot_strategy_by_year

# combined is the full z-scored df already built in the notebook
strategy = CompositeSignalIndex(full_zscored_df=combined)
strategy.fit(train_df)   # runs Nelder-Mead, stores opt_logits / opt_exponent

# Build the combined eval df (train + test full windows only)
full_eval_df = pd.concat([train_df, test_df], ignore_index=True)

Starting Nelder-Mead optimisation …

  eval   500 | SPD ratio 1.282699 | γ=20.0855
  eval  1000 | SPD ratio 1.287466 | γ=20.0855

Optimisation complete
  Status         : Optimization terminated successfully.
  Function evals : 1494
  Best SPD ratio : 1.290656  (+29.07% vs DCA)
  γ              : 20.0855

Learned signal weights (softmax):
  mvrv                                          0.2011
  sopr_7d_ema                                   0.0868
  reserve_risk                                  0.0197
  greed_index                                   0.0626
  puell_multiple                                0.0488
  lth_nupl                                      0.1579
  sell_side_risk_ratio_7d_ema                   0.0098
  net_unrealized_pnl_rel_to_market_cap          0.4132


In [12]:
from stacksats import StrategyRunner, BacktestConfig, ExportConfig

btc_df_pl = (
    pl.from_pandas(
        full_eval_df[["date", "price_usd"]]
        .assign(date=lambda d: pd.to_datetime(d["date"]).dt.date)
    )
    .with_columns(pl.col("date").cast(pl.Date))
)

runner = StrategyRunner()
start_date = str(full_eval_df["date"].min().date())
end_date   = str(full_eval_df["date"].max().date())

csi_bt = runner.backtest(
    strategy,
    config=BacktestConfig(
        start_date=start_date,
        end_date=end_date,
    ),
    btc_df=btc_df_pl,
)

csi_bt = csi_bt.to_dataframe()

In [13]:
from stacksats import StrategyRunner, ComparisonConfig, UniformStrategy

csi_comparison = runner.compare(
    strategies=[strategy, UniformStrategy()],
    config=ComparisonConfig(
        start_date=start_date,
        end_date=end_date,
        baseline="uniform",
        strict=False,
        output_dir="output",
    ),
    btc_df=btc_df_pl,
)

csi_comparison.to_dataframe()

selector,strategy_id,strategy_version,intent_mode,tier,promotion_stage,validation_passed,judgment_label,win_rate,score,exp_decay_percentile,multiple_vs_uniform,score_delta_vs_baseline,exp_decay_delta_vs_baseline,is_baseline
str,str,str,str,str,str,bool,str,f64,f64,f64,f64,f64,f64,bool
"""composite-signal-index""","""composite-signal-index""","""1.0.0""","""propose""",null,null,true,"""validation-passed""",58.326818,60.824291,63.321764,1.665384,41.813129,25.299441,false
"""uniform""","""uniform""","""1.0.0""","""propose""","""stable""","""promoted""",false,"""validation-failed""",0.0,19.011162,38.022323,1.0,0.0,0.0,true


In [14]:
from stacksats import StrategyRunner, ComparisonConfig, UniformStrategy

csi_yearly = []
uniform_yearly = []
uniform_strategy = UniformStrategy()

for year in range(2018, 2026):
    csi = strategy_utils.export_one_year(strategy, btc_df_pl, year, runner)
    if csi is not None:
        csi_yearly.append(csi)

    uniform_result = strategy_utils.export_one_year(uniform_strategy, btc_df_pl, year, runner)
    if uniform_result is not None:
        uniform_yearly.append(uniform_result)

print("CSI valid yearly exports:", len(csi_yearly))
print("Uniform valid yearly exports:", len(uniform_yearly))
if not csi_yearly:
    raise ValueError("No valid CSI exports were produced.")

if not uniform_yearly:
    raise ValueError("No valid Uniform exports were produced.")

csi_all = pl.concat(csi_yearly).rename({"weight": "csi_weight_raw"})
uniform_all = pl.concat(uniform_yearly).rename({"weight": "baseline_weight_raw"})

print("CSI combined rows:", csi_all.height)
print("Uniform combined rows:", uniform_all.height)
merged = (
    csi_all
    .select(["date", "price_usd", "csi_weight_raw"])
    .join(
        uniform_all.select(["date", "baseline_weight_raw"]),
        on="date",
        how="inner"
    )
    .sort("date")
)

print("Merged rows:", merged.height)

merged.head()
csi_sum = merged["csi_weight_raw"].sum()
uniform_sum = merged["baseline_weight_raw"].sum()

merged = merged.with_columns([
    pl.col("csi_weight_raw").alias("csi_weight"),
    pl.col("baseline_weight_raw").alias("baseline_weight"),
])

print("CSI normalized weight sum:", merged["csi_weight"].sum())
print("Uniform normalized weight sum:", merged["baseline_weight"].sum())

merged = merged.with_columns([
    (pl.col("csi_weight") * total_budget_usd).alias("csi_usd"),
    (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
])

merged = merged.with_columns([
    (pl.col("csi_usd") / pl.col("price_usd")).alias("btc_accum_csi"),
    (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
])

merged = merged.with_columns([
    (pl.col("btc_accum_csi") * 100_000_000).alias("sats_accum_csi"),
    (pl.col("btc_accum_baseline") * 100_000_000).alias("sats_accum_baseline"),
])

merged = merged.with_columns([
    (pl.col("sats_accum_csi") / pl.col("csi_usd")).alias("sats_per_dollar_csi"),
    (pl.col("sats_accum_baseline") / pl.col("baseline_usd")).alias("sats_per_dollar_baseline"),
])

merged = merged.with_columns(pl.col("date").cast(pl.Date).dt.year().alias("year"))

spd_by_year = (
    merged
    .group_by("year")
    .agg([
        (pl.col("csi_weight") / pl.col("price_usd")).sum().alias("spd_csi"),
        (pl.col("baseline_weight") / pl.col("price_usd")).sum().alias("spd_baseline"),
    ])
    .with_columns(
        ((pl.col("spd_csi") / pl.col("spd_baseline") - 1) * 100).alias("improvement_pct")
    )
    .sort("year")
)
print(spd_by_year)

n_years = merged.select(pl.col("year").n_unique()).item()
total_spend_usd = total_budget_usd * n_years   # $1000 × 8 = $8000

total_dynamic_btc = merged["btc_accum_csi"].sum()
total_baseline_btc = merged["btc_accum_baseline"].sum()

sats_per_dollar_dynamic = (total_dynamic_btc / total_spend_usd) * 100_000_000
sats_per_dollar_baseline = (total_baseline_btc / total_spend_usd) * 100_000_000

pct_diff_vs_baseline = (
    (total_dynamic_btc - total_baseline_btc) / total_baseline_btc
) * 100

performance_label = "better" if pct_diff_vs_baseline > 0 else "worse"

print(f"Total BTC accumulated, CSI: {total_dynamic_btc:.6f}  (${total_spend_usd:.0f} deployed)")
print(f"Total BTC accumulated, Uniform: {total_baseline_btc:.6f} ")
print(f"Sats per dollar, CSI: {sats_per_dollar_dynamic:.2f}")
print(f"Sats per dollar, Uniform: {sats_per_dollar_baseline:.2f}")
print(f"CSI performed {abs(pct_diff_vs_baseline):.2f}% {performance_label} than Uniform")

merged = merged.with_columns(pl.col("date").cast(pl.Date).dt.year().alias("year"))

plot_df = merged.to_pandas()
plot_df["date"] = pd.to_datetime(plot_df["date"])
plot_df["year"] = plot_df["date"].dt.year
#plot_df["cycle_label"] = plot_df["date"].apply(plots.assign_cycle_label)

plot_df.head()
top_buy_points_list = []

for year, year_df in plot_df.groupby("year"):
    threshold = year_df["csi_weight"].quantile(0.90)

    top_year_df = year_df[year_df["csi_weight"] >= threshold].copy()
    top_year_df["top_buy_threshold_year"] = threshold

    top_buy_points_list.append(top_year_df)

top_buy_points = pd.concat(top_buy_points_list, ignore_index=True)

top_buy_points = top_buy_points.sort_values(
    ["year", "csi_weight"],
    ascending=[True, False]
)

print("Top buy points:", len(top_buy_points))
top_buy_points[["date", "year", "price_usd", "csi_weight", "top_buy_threshold_year"]].head()

cols = plots.StrategyColumns(
    weight="csi_weight",
    spd="sats_per_dollar_csi",
    sats_accum="sats_accum_csi",
)

# Full period
full_plot = plots.plot_strategy_full_period(
    plot_df, 
    cols, 
    "CSI Strategy", 
    date_range=("2018-01-01", "2025-12-31"),
    test_start_date="2024-01-01",
)
plt.show()

# yearly breakdown
yearly_plot = plots.plot_strategy_by_year(
    plot_df,
    cols,
    "CSI Strategy — Yearly Breakdown"
)
plt.show()

2018: exported 365 rows
2018: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows
2020: exported 365 rows
2020: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2022: exported 365 rows
2022: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
2024: exported 365 rows
2024: exported 365 rows
2025: exported 365 rows
2025: exported 365 rows
CSI valid yearly exports: 8
Uniform valid yearly exports: 8
CSI combined rows: 2920
Uniform combined rows: 2920
Merged rows: 2920
CSI normalized weight sum: 8.000000000000028
Uniform normalized weight sum: 8.000000000000021
shape: (8, 4)
┌──────┬──────────┬──────────────┬─────────────────┐
│ year ┆ spd_csi  ┆ spd_baseline ┆ improvement_pct │
│ ---  ┆ ---      ┆ ---          ┆ ---             │
│ i32  ┆ f64      ┆ f64          ┆ f64             │
╞══════╪══════════╪══════════════╪═════════════════╡
│ 2018 ┆ 0.000187 ┆ 0.000147     ┆ 27.103227       │
│ 2019 ┆ 0.000151 ┆ 0.000159     ┆ -5.167536       │
│ 2020 